In [ ]:
import numpy as np
from easyscience.variable import Parameter
from scipy.special import voigt_profile

from easydynamics.sample_model import DeltaFunction, Gaussian, Lorentzian, SampleModel
from easydynamics.utils import convolution as convolution

# Numerical convolutions are not very accurate
NUMERICAL_CONVOLUTION_ABSOLUTE_TOLERANCE = 1e-6
NUMERICAL_CONVOLUTION_RELATIVE_TOLERANCE = 1e-5



# WHEN
sample_lorentzian = Lorentzian(
    center=0.1, width=0.3, area=2, name="SampleLorentzian"
)
sample_delta = DeltaFunction(center=0.5, area=4, name="SampleDelta")
resolution_gauss = Gaussian(
    center=-0.3, width=0.4, area=3, name="ResolutionGauss"
)
sample = SampleModel(name="SampleModel")
sample.add_component(sample_lorentzian)
sample.add_component(sample_delta)
resolution = SampleModel(name="ResolutionModel")
resolution.add_component(resolution_gauss)

# THEN
x = np.linspace(-10, 10, 20001)
calculated_convolution = convolution(
    x=x,
    sample_model=sample,
    resolution_model=resolution,
    method="numerical",
    upsample_factor=5,
)

# EXPECT: Combine Gaussian, Lorentzian, and Delta functions contributions
expected_voigt = 2 * 3 * voigt_profile(x - (0.1 - 0.3), 0.4, 0.3)
expected_gauss_center = -0.3 + 0.5
expected_gauss = (
    3
    * 4
    * np.exp(-0.5 * ((x - (expected_gauss_center)) / 0.4) ** 2)
    / (np.sqrt(2 * np.pi) * 0.4)
)
expected_result = expected_voigt + expected_gauss

import matplotlib.pyplot as plt
%matplotlib widget
plt.plot(x, calculated_convolution, label="Convolution Result")
plt.plot(x, expected_result, label="Expected Result", linestyle='dashed')

In [ ]:
print(calculated_convolution)